In [ ]:
# ── Cell 1: Imports & Paths ───────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import copy

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline   # replaces bare SMOTE usage

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = Path.cwd()
PROJECT_ROOT  = NOTEBOOK_DIR.parent
DATA_PATH     = PROJECT_ROOT / "data" / "processed" / "preprocessed.csv"
MODELS_PATH   = PROJECT_ROOT / "models"
MODELS_PATH.mkdir(exist_ok=True)

RANDOM_STATE  = 42

print(f"Data   : {DATA_PATH}")
print(f"Models : {MODELS_PATH}")

In [ ]:
# ── Cell 2: Load & Validate ───────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)

print(f"Shape  : {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

# Hard-check that the corrected preprocessing was run
REQUIRED_COLS = [
    "log_stellar_flux",
    "log_surface_gravity",
    "bulk_density_gcc",
    "habitable_binary",
    "label_is_measured",    # created in main.ipynb before any imputation
]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise RuntimeError(
        f"\n{'='*60}\n"
        f"STOP: The following required columns are absent:\n  {missing}\n"
        f"This means you are loading the OLD preprocessed CSV.\n"
        f"Re-run the corrected main.ipynb first, then come back here.\n"
        f"{'='*60}"
    )

print("\nAll required columns present. Correct CSV confirmed.")

In [ ]:
# ── Cell 3: Train on ALL 6,100 rows, test on measured-label rows only ─────────────────
measured = df[df["label_is_measured"] == 1].copy()   # 1,508 rows — trusted labels
infer_df = df[df["label_is_measured"] == 0].copy()   # 4,592 rows — score only

# 20% of measured rows → clean test set
meas_train, meas_test = train_test_split(
    measured, test_size=0.2, random_state=RANDOM_STATE,
    stratify=measured["habitable_binary"],
)

# Training pool = measured train rows + ALL imputed-label rows
train_df = pd.concat([meas_train, infer_df], ignore_index=True)

print(f"Training pool : {len(train_df):,} rows  ({int(train_df['habitable_binary'].sum())} positives)")
print(f"Test set      : {len(meas_test):,} rows  ({int(meas_test['habitable_binary'].sum())} positives)  ← real labels only")
print(f"Inference pool: {len(infer_df):,} rows  (no labels used)")

In [ ]:
# ── Cell 4: Feature Selection ─────────────────────────────────────────────────
#
# FEATURES INCLUDED
# ─────────────────────────────────────────────────────────────────────────────
# radius_earth       — planet size (rocky planet range 0.5–1.6 defines target
#                      but is a real astrophysical predictor; see NOTE below)
# mass_earth         — planet mass
# orbital_period     — proxy for orbital distance
# semimajor_axis     — direct star-planet distance [AU]
# star_temp_k        — stellar effective temperature (continuous class proxy)
# star_luminosity    — log10(L/L_sun)
# star_metallicity   — planet formation likelihood
# log_stellar_flux   — KEY: habitable zone position, no leakage from eq_temp_k
# log_surface_gravity— atmosphere retention proxy
# bulk_density_gcc   — rocky vs gaseous discriminator
#
# FEATURES EXCLUDED
# ─────────────────────────────────────────────────────────────────────────────
# eq_temp_k          — directly defines the target → HARD LEAKAGE, must drop
# density            — independently imputed, physically inconsistent version
# star_spectype      — redundant string; star_temp_k is the continuous version
# star_class         — same redundancy as star_spectype
# planet_name        — identifier
# host_star_name     — identifier
# label_is_measured  — bookkeeping column, not a feature
# habitable_binary   — this is y
#
# NOTE on radius_earth
# ─────────────────────────────────────────────────────────────────────────────
# The target includes a radius threshold (0.5–1.6 R_earth). Including
# radius_earth in X means the model will partly learn that threshold.
# However, the threshold is astrophysically motivated (rocky planet range),
# and radius is genuinely predictive. We keep it but note the caveat.
# To run a cleaner experiment, remove it from X_COLS and compare results.
#
# Add star_class dummies to X_COLS if main.ipynb added them:
STAR_DUMMY_COLS = [c for c in df.columns if c.startswith("star_class_")]

X_COLS = [
    "radius_earth",
    "mass_earth",
    "orbital_period",
    "semimajor_axis",
    "star_temp_k",
    "star_luminosity",
    "star_metallicity",
    "log_stellar_flux",
    "log_surface_gravity",
    "bulk_density_gcc",
] + STAR_DUMMY_COLS   # ← adds the one-hot columns from main.ipynb

DROP_COLS = [
    "eq_temp_k",         # defines target — MUST drop
    "density",           # inconsistently imputed
    "star_spectype",     # redundant
    "star_class",        # redundant
    "planet_name",       # identifier
    "host_star_name",    # identifier
    "label_is_measured", # bookkeeping
    "habitable_binary",  # this is y
]

# Verify all X_COLS are present
missing_x = [c for c in X_COLS if c not in train_df.columns]
if missing_x:
    raise RuntimeError(f"Missing feature columns: {missing_x}")

print(f"Feature count : {len(X_COLS)}")
print(f"Features      : {X_COLS}")
print(f"Star dummies   : {STAR_DUMMY_COLS}")
print(f"\nLabel distribution (training pool):\n{train_df['habitable_binary'].value_counts()}")
print(f"Class imbalance ratio: {train_df['habitable_binary'].value_counts()[0] / train_df['habitable_binary'].value_counts()[1]:.1f}:1")

In [ ]:
# ── Cell 5: Update X/y to use the new train_df / meas_test ────────────────────────
X_train = train_df[X_COLS]
y_train = train_df["habitable_binary"]
X_test  = meas_test[X_COLS]
y_test  = meas_test["habitable_binary"]

n_pos_train = int(y_train.sum())
print(f"Train: {X_train.shape[0]:,} rows, {n_pos_train} positives")
print(f"Test : {X_test.shape[0]:,} rows, {int(y_test.sum())} positives")

In [ ]:
# ── Cell 6: Fix the SMOTE-inside-CV bug (most critical fix) ───────────────────────
#
# PROBLEM IN YOUR ORIGINAL CODE:
#   Cell 7 ran SMOTE on ALL of X_train → produced X_train_res
#   Cell 9 ran CV on X_train_res
#   → validation folds contained near-duplicate synthetic points
#   → XGBoost CV AUPRC = 1.0000 was a direct result of this bug
#
# FIX: SMOTE lives inside the pipeline. CV never sees synthetic validation data.
#
k_neighbors = max(1, min(5, n_pos_train - 1))
neg_ratio   = int((y_train == 0).sum()) / max(n_pos_train, 1)

def make_smote():
    return SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors,
                 sampling_strategy=0.2)

# Each pipeline handles its own scaling — no global scaler needed.
all_pipelines = {
    "Logistic Regression": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote",  make_smote()),
        ("model",  LogisticRegression(
            random_state=RANDOM_STATE, max_iter=2000,
            class_weight="balanced", C=0.1)),
    ]),
    "SVM": ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote",  make_smote()),
        ("model",  SVC(probability=True, random_state=RANDOM_STATE,
                       class_weight="balanced", C=1.0, kernel="rbf")),
    ]),
    "Decision Tree": ImbPipeline([
        ("smote", make_smote()),
        ("model", DecisionTreeClassifier(
            random_state=RANDOM_STATE, class_weight="balanced",
            max_depth=5, min_samples_leaf=3)),
    ]),
    "Random Forest": ImbPipeline([
        ("smote", make_smote()),
        ("model", RandomForestClassifier(
            random_state=RANDOM_STATE, class_weight="balanced",
            n_estimators=200, max_depth=8, min_samples_leaf=2)),
    ]),
    "XGBoost": ImbPipeline([
        ("smote", make_smote()),
        ("model", xgb.XGBClassifier(
            random_state=RANDOM_STATE, eval_metric="aucpr",
            scale_pos_weight=neg_ratio, n_estimators=200, max_depth=4,
            learning_rate=0.05, subsample=0.8, use_label_encoder=False)),
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results       = []
trained_pipes = {}
test_probs    = {}

for name, pipe in all_pipelines.items():
    print(f"\n{'='*55}\n  {name}\n{'='*55}")

    # CV on RAW X_train — SMOTE runs inside each fold, never touches val fold
    cv_res = cross_validate(
        copy.deepcopy(pipe), X_train, y_train,
        cv=cv, scoring="average_precision",
        return_train_score=False, n_jobs=-1,
    )
    cv_mean = cv_res["test_score"].mean()
    cv_std  = cv_res["test_score"].std()

    # Fit on full training pool
    pipe.fit(X_train, y_train)

    # Evaluate on clean measured-label test set
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)

    auprc = average_precision_score(y_test, y_prob)
    roc   = roc_auc_score(y_test, y_prob)
    f1    = f1_score(y_test, y_pred, zero_division=0)
    cm    = confusion_matrix(y_test, y_pred)

    print(f"  Test AUPRC  : {auprc:.4f}  ← measured labels only")
    print(f"  Test ROC-AUC: {roc:.4f}")
    print(f"  Test F1     : {f1:.4f}")
    print(f"  CV AUPRC    : {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"\n  Confusion Matrix:\n{cm}")
    print(f"\n{classification_report(y_test, y_pred, target_names=['Non-Hab','Hab'], zero_division=0)}")

    trained_pipes[name] = pipe
    test_probs[name]    = y_prob
    results.append(dict(
        Model=name, Test_AUPRC=round(auprc,4), Test_ROC=round(roc,4),
        Test_F1=round(f1,4), CV_AUPRC=round(cv_mean,4), CV_std=round(cv_std,4),
    ))
    joblib.dump(pipe, MODELS_PATH / f"{name.lower().replace(' ','_')}.pkl")

results_df = pd.DataFrame(results).sort_values("Test_AUPRC", ascending=False)
print(f"\n{'='*55}\nFINAL RESULTS:")
print(results_df.to_string(index=False))

In [ ]:
# ── Cell 7: Ensemble (fixed scaling issue) ───────────────────────────────────────
# PROBLEM: original VotingClassifier received pre-scaled X_test for tree models
# FIX: each sub-pipeline scales internally; VotingClassifier gets raw data
top3 = results_df.head(3)["Model"].tolist()

ensemble = VotingClassifier(
    estimators=[
        (n.lower().replace(" ","_"), copy.deepcopy(all_pipelines[n]))
        for n in top3
    ],
    voting="soft",
)
ensemble.fit(X_train, y_train)   # raw data — pipelines handle scaling

y_prob_ens = ensemble.predict_proba(X_test)[:, 1]
y_pred_ens = ensemble.predict(X_test)
print(f"Ensemble  AUPRC={average_precision_score(y_test,y_prob_ens):.4f}"
      f"  ROC={roc_auc_score(y_test,y_prob_ens):.4f}"
      f"  F1={f1_score(y_test,y_pred_ens,zero_division=0):.4f}")

trained_pipes["Ensemble"] = ensemble
test_probs["Ensemble"]    = y_prob_ens
joblib.dump(ensemble, MODELS_PATH / "ensemble.pkl")

In [ ]:
# ── Cell 8: Precision-Recall Curves ───────────────────────────────────────────────
#
# PR curves are the correct visualisation for imbalanced binary classification.
# A random classifier baseline sits at precision = positive_rate ≈ 0.004.
# Any meaningful model should be substantially above this baseline.
#
pos_rate = y_test.mean()

fig, ax = plt.subplots(figsize=(9, 6))

colors = plt.cm.tab10.colors
for idx, (name, probs) in enumerate(test_probs.items()):
    p, r, _ = precision_recall_curve(y_test, probs)
    auprc = average_precision_score(y_test, probs)
    ax.step(r, p, where="post", color=colors[idx % 10],
            label=f"{name}  (AUPRC={auprc:.3f})", linewidth=1.8)

ax.axhline(pos_rate, color="gray", linestyle="--", linewidth=1,
           label=f"Random baseline ({pos_rate:.3f})")

ax.set_xlabel("Recall",    fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curves\n(primary metric for imbalanced classification)",
             fontsize=12)
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_PATH / "pr_curves.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 9: Feature Importance (fixed access to wrapped models) ───────────────────
# PROBLEM: trained_models[name].feature_importances_ — models are now wrapped
# FIX: access via named_steps["model"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mname, color in zip(axes, ["Random Forest","XGBoost"], ["steelblue","tomato"]):
    inner_model = trained_pipes[mname].named_steps["model"]   # ← the fix
    importances = inner_model.feature_importances_
    fi = (pd.DataFrame({"Feature": X_COLS, "Importance": importances})
            .sort_values("Importance", ascending=True))
    ax.barh(fi["Feature"], fi["Importance"], color=color)
    ax.set_title(f"{mname} — Feature Importance")
    ax.set_xlabel("Importance"); ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_PATH / "feature_importance.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 10: Candidate Scoring (fixed scaling issue) ───────────────────────────
# ORIGINAL: X_infer_scaled = scaler.transform(X_infer)  ← scaler no longer exists
# FIX: pipelines handle scaling internally
if len(infer_df) > 0:
    best_name = results_df.iloc[0]["Model"]
    best_pipe = trained_pipes[best_name]
    infer_prob = best_pipe.predict_proba(infer_df[X_COLS])[:, 1]   # raw data, no scaler
    cands = infer_df[["planet_name","host_star_name"]].copy()
    cands["habitability_prob"] = infer_prob.round(4)
    cands = cands.sort_values("habitability_prob", ascending=False)
    print(f"\nTop-20 candidates ({best_name}):")
    print(cands.head(20).to_string(index=False))
    cands.to_csv(MODELS_PATH / "candidate_planets.csv", index=False)
else:
    print("No inference rows — all planets had measured labels.")

In [ ]:
# ── Cell 11: Save Results ───────────────────────────────────────────────────────
results_df.to_csv(MODELS_PATH / "model_results.csv", index=False)
print("Saved: models/model_results.csv")

print(f"\n{'='*55}")
print("ALL MODELS SAVED")
for f in sorted(MODELS_PATH.iterdir()):
    print(f"  {f.name}")
print(f"{'='*55}")

In [ ]:
# ── Cell 12: Interpretation Notes ───────────────────────────────────────────────────
NOTES = """
╔══════════════════════════════════════════════════════════════════════╗
║  INTERPRETATION GUIDE                                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PRIMARY METRIC: AUPRC (Area Under Precision-Recall Curve)          ║
║  ─────────────────────────────────────────────────────────────────  ║
║  • Do NOT use accuracy — all-zeros classifier = 99.6% accurate.     ║
║  • AUPRC random baseline ≈ positive_rate ≈ 0.004 (~0.4%)            ║
║  • A model scoring AUPRC > 0.1 is performing well above chance.     ║
║  • ROC-AUC is reported but is optimistic under this level of         ║
║    imbalance — use it as a secondary indicator only.                 ║
║                                                                      ║
║  STATISTICAL RELIABILITY WARNING                                     ║
║  ─────────────────────────────────────────────────────────────────  ║
║  • ~20 positives in training, ~5 in test.                            ║
║  • One misclassification changes F1 from 0.91 to 0.67.              ║
║  • CV std tells you more than CV mean here.                          ║
║  • A single run of this notebook may give very different numbers.    ║
║  • Results are indicative, not statistically conclusive.             ║
║                                                                      ║
║  LEAKAGE STATUS                                                      ║
║  ─────────────────────────────────────────────────────────────────  ║
║  • eq_temp_k excluded from X (was half the target definition).       ║
║  • log_stellar_flux is safe: derived from luminosity + semimajor_    ║
║    axis, not from eq_temp_k (which is excluded from MICE input).     ║
║  • radius_earth is kept with the caveat that it is also part of      ║
║    the target threshold. Remove it to run a purer experiment.        ║
║                                                                      ║
║  TRAINING STRATEGY UPDATE                                            ║
║  ─────────────────────────────────────────────────────────────────  ║
║  • Training on ALL 6,100 rows (measured + imputed labels)             ║
║  • Testing ONLY on measured labels (clean evaluation)                ║
║  • SMOTE inside CV prevents synthetic data leakage                    ║
║  • More realistic performance estimates                               ║
║                                                                      ║
║  CANDIDATE SCORING                                                   ║
║  ─────────────────────────────────────────────────────────────────  ║
║  • candidate_planets.csv contains planets whose eq_temp_k or radius  ║
║    was missing, so they could not be reliably labelled.              ║
║  • Their probability scores are exploratory — treat as a ranked      ║
║    list of "planets worth investigating", not confirmed habitability. ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(NOTES)

In [ ]:
# ── Cell 13: End of Notebook ───────────────────────────────────────────────────────
print("ML pipeline completed successfully!")